# M2 — U-Net sâu tự xây từ đầu
Run All theo thứ tự: dữ liệu → model → train → biểu đồ → test.
Chỉ đổi `SEED` ở ô đầu để chạy một seed khác.


In [ ]:
import csv
import json
import random
import shutil
import sys
import tempfile
import time
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import SimpleITK as sitk
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchinfo import summary

SEED = 42  # Một lần Run All = một lần train đầy đủ với seed này.
IN_NOTEBOOK = True

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(errors="backslashreplace")
    sys.stderr.reconfigure(errors="backslashreplace")

ROOT = Path.cwd()
if not (ROOT / "data" / "splits_v1.csv").exists():
    ROOT = ROOT.parent
CACHE = ROOT / "data" / "processed" / "flair_wt_v1"
CACHE.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("Cần PyTorch CUDA trong Windows .venv để train/test cuối.")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"Seed: {SEED} | Thiết bị: {DEVICE}")
print(f"Python: {sys.executable} | PyTorch: {torch.__version__}")


## 1. Nạp và chuẩn bị dữ liệu
Mỗi ca có một ảnh FLAIR 3D và một mask OT 3D. Bảng `splits_v1.csv` chia
**theo bệnh nhân**, nên 32 lát của cùng một ca luôn nằm trong cùng một tập.
Hàm `read_case` thực hiện cùng một công thức tiền xử lý cho train/val/test.


In [ ]:
with (ROOT / "data" / "splits_v1.csv").open(encoding="utf-8", newline="") as file:
    cases = list(csv.DictReader(file))
train_cases = [row for row in cases if row["split"] == "train"]
val_cases = [row for row in cases if row["split"] == "val"]
test_cases = [row for row in cases if row["split"] == "test"]
assert (len(train_cases), len(val_cases), len(test_cases)) == (70, 15, 15)
assert len({row["case_id"] for row in cases}) == 100
print(f"BraTS 2015: {len(train_cases)} train | {len(val_cases)} validation | {len(test_cases)} test")
print("HGG/LGG:", dict(Counter(row["grade"] for row in cases)))


def read_case(row):
    """Đọc một ca và trả về 32 lát FLAIR/WT, mỗi lát 128×128."""
    # SimpleITK trên máy này cần đường dẫn tạm không có dấu tiếng Việt.
    with tempfile.TemporaryDirectory(prefix="brats_mha_") as folder:
        flair_file, ot_file = Path(folder) / "flair.mha", Path(folder) / "ot.mha"
        shutil.copyfile(ROOT / row["flair_relpath"], flair_file)
        shutil.copyfile(ROOT / row["mask_relpath"], ot_file)
        flair = sitk.GetArrayFromImage(sitk.ReadImage(str(flair_file))).astype(np.float32)
        ot = sitk.GetArrayFromImage(sitk.ReadImage(str(ot_file)))
    if flair.shape != ot.shape:
        raise ValueError(f"FLAIR/OT khác shape: {row['case_id']}")

    # Chuẩn hóa theo median/IQR của voxel FLAIR khác 0, nền giữ bằng 0.
    brain = flair != 0
    median = np.median(flair[brain])
    q25, q75 = np.percentile(flair[brain], (25, 75))
    flair = np.clip((flair - median) / max(q75 - q25, 1e-6), -5, 5)
    flair = (flair + 5) / 10
    flair[~brain] = 0

    # Vị trí lát chỉ phụ thuộc độ sâu ảnh; không nhìn OT để chọn lát.
    depth = flair.shape[0]
    indices = np.rint(np.linspace(0.2 * (depth - 1), 0.8 * (depth - 1), 32)).astype(int)
    x = torch.from_numpy(flair[indices].copy()).unsqueeze(1)
    y = torch.from_numpy(np.isin(ot[indices], (1, 2, 3, 4)).astype(np.float32)).unsqueeze(1)
    x = F.interpolate(x, size=(128, 128), mode="bilinear", align_corners=False)
    y = F.interpolate(y, size=(128, 128), mode="nearest")
    return x[:, 0].numpy().astype(np.float16), y[:, 0].numpy().astype(np.uint8)


def load_case(row):
    """Đọc cache chung; nếu chưa có thì tiền xử lý và lưu lại."""
    path = CACHE / (row["case_id"].replace("/", "__") + ".npz")
    if not path.exists():
        image, mask = read_case(row)
        np.savez_compressed(path, image=image, mask=mask)
    with np.load(path) as saved:
        return saved["image"], saved["mask"]


sample_image, sample_mask = load_case(train_cases[0])
print("Một ca sau xử lý — FLAIR:", sample_image.shape, "| WT:", sample_mask.shape)
if IN_NOTEBOOK:
    slice_index = 16
    fig, axes = plt.subplots(1, 2, figsize=(7, 3))
    axes[0].imshow(sample_image[slice_index], cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("FLAIR")
    axes[1].imshow(sample_mask[slice_index], cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("WT mask")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(f"{train_cases[0]['case_id']} — lát {slice_index + 1}/32")
    fig.tight_layout()
    plt.show()
    plt.close(fig)


## 2. Đưa các lát ảnh vào DataLoader
`SliceDataset` trả `(ảnh, mask, case_id)`. Chỉ train mới lật ngang đồng bộ.
FLAIR được lặp thành 3 kênh và chuẩn hóa như M2/M3.


In [ ]:
MEAN = torch.tensor((0.485, 0.456, 0.406))[:, None, None]
STD = torch.tensor((0.229, 0.224, 0.225))[:, None, None]


class SliceDataset(Dataset):
    def __init__(self, rows, augment=False):
        pairs = [load_case(row) for row in rows]
        self.images = np.concatenate([x for x, _ in pairs])
        self.masks = np.concatenate([y for _, y in pairs])
        self.case_ids = [row["case_id"] for row in rows for _ in range(32)]
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = torch.from_numpy(self.images[index].astype(np.float32))[None]
        mask = torch.from_numpy(self.masks[index].astype(np.float32))[None]
        if self.augment and torch.rand(()) < 0.5:
            image, mask = image.flip(-1), mask.flip(-1)
        image = (image.repeat(3, 1, 1) - MEAN) / STD
        return image, mask, self.case_ids[index]


train_data = SliceDataset(train_cases, augment=True)
val_data = SliceDataset(val_cases)
train_loader = DataLoader(train_data, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_data, batch_size=16, shuffle=False, num_workers=0)
print(f"Số lát dùng: {len(train_data)} train | {len(val_data)} validation")


## 3. U-Net sâu M2 tự viết
Encoder giảm kích thước bốn lần; decoder ghép bốn skip để giữ chi tiết.
Toàn bộ lớp học từ đầu, không tải model hay weights dựng sẵn.


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.layers(x)


class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x):
        return self.conv(F.max_pool2d(x, 2))


class UpBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat((x, skip), dim=1))


class ScratchUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = DoubleConv(3, 32)
        self.enc2 = DownBlock(32, 64)
        self.enc3 = DownBlock(64, 128)
        self.enc4 = DownBlock(128, 256)
        self.middle = DownBlock(256, 512)
        self.dec4 = UpBlock(512 + 256, 256)
        self.dec3 = UpBlock(256 + 128, 128)
        self.dec2 = UpBlock(128 + 64, 64)
        self.dec1 = UpBlock(64 + 32, 32)
        self.head = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        x = self.middle(e4)
        x = self.dec4(x, e4)
        x = self.dec3(x, e3)
        x = self.dec2(x, e2)
        x = self.dec1(x, e1)
        return self.head(x)


def loss_fn(logits, mask):
    bce = F.binary_cross_entropy_with_logits(logits, mask)
    probability = torch.sigmoid(logits.float())
    intersection = (probability * mask).sum(dim=(1, 2, 3))
    total = probability.sum(dim=(1, 2, 3)) + mask.sum(dim=(1, 2, 3))
    soft_dice = 1 - ((2 * intersection + 1) / (total + 1)).mean()
    return 0.5 * bce + 0.5 * soft_dice


model = ScratchUNet().to(DEVICE)
print(summary(model, input_data=torch.zeros(1, 3, 128, 128, device=DEVICE),
              depth=1, row_settings=("var_names",), mode="eval", verbose=0))


## 4. Huấn luyện và chọn checkpoint bằng validation
Một AdamW, giảm learning rate và dừng sớm theo Dice validation.
Mỗi epoch in loss, Dice theo bệnh nhân và ngưỡng đã chọn.


In [ ]:
def patient_scores(prediction, truth):
    tp = int(np.logical_and(prediction, truth).sum())
    fp = int(np.logical_and(prediction, ~truth).sum())
    fn = int(np.logical_and(~prediction, truth).sum())
    tn = int(np.logical_and(~prediction, ~truth).sum())
    if tp + fp + fn == 0:
        return {"dice": 1.0, "iou": 1.0, "precision": 1.0,
                "recall": 1.0, "pixel_accuracy": 1.0}
    return {
        "dice": 2 * tp / max(2 * tp + fp + fn, 1),
        "iou": tp / max(tp + fp + fn, 1),
        "precision": tp / max(tp + fp, 1),
        "recall": tp / max(tp + fn, 1),
        "pixel_accuracy": (tp + tn) / (tp + fp + fn + tn),
    }


@torch.inference_mode()
def predict_by_patient(loader):
    model.eval()
    probabilities, masks = defaultdict(list), defaultdict(list)
    for images, truth, case_ids in loader:
        predicted = torch.sigmoid(model(images.to(DEVICE))).cpu().numpy()
        for i, case_id in enumerate(case_ids):
            probabilities[case_id].append(predicted[i, 0])
            masks[case_id].append(truth[i, 0].numpy().astype(bool))
    return ({key: np.stack(value) for key, value in probabilities.items()},
            {key: np.stack(value) for key, value in masks.items()})


def score_at_threshold(probabilities, masks, threshold):
    return [{"case_id": case_id, "grade": case_id.split("/")[0],
             **patient_scores(probabilities[case_id] >= threshold, masks[case_id])}
            for case_id in sorted(probabilities)]


def save_table(path, records):
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)


THRESHOLDS = (0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70)


In [ ]:
run_dir = ROOT / "runs" / "m2" / str(SEED)
run_dir.mkdir(parents=True, exist_ok=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-5)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
history = []
best_dice, best_epoch, best_threshold = -1.0, 0, 0.5
no_improvement = 0
max_epochs = 30
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()

for epoch in range(1, max_epochs + 1):
    model.train()
    total_loss = 0.0
    for batch_index, (images, masks, _) in enumerate(train_loader):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", enabled=DEVICE.type == "cuda"):
            loss = loss_fn(model(images), masks)
        scaler.scale(loss).backward()
        if epoch == 1 and batch_index == 0:
            print("Encoder M2 nhận gradient:", model.enc1.layers[0].weight.grad is not None)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    probabilities, truths = predict_by_patient(val_loader)
    candidates = []
    for threshold in THRESHOLDS:
        records = score_at_threshold(probabilities, truths, threshold)
        candidates.append((np.mean([row["dice"] for row in records]), threshold, records))
    val_dice, threshold, val_records = max(candidates, key=lambda x: (x[0], -abs(x[1] - 0.5)))
    learning_rate = optimizer.param_groups[0]["lr"]
    history.append({"epoch": epoch, "train_loss": total_loss / len(train_loader),
                    "val_dice": float(val_dice), "threshold": threshold,
                    "learning_rate": learning_rate})
    save_table(run_dir / "history.csv", history)
    print(f"Epoch {epoch:02d}/{max_epochs} | loss {history[-1]['train_loss']:.4f} | "
          f"val Dice {val_dice:.4f} | ngưỡng {threshold:.2f} | lr {learning_rate:.2g}")

    if val_dice > best_dice + 1e-4:
        best_dice, best_epoch, best_threshold = float(val_dice), epoch, threshold
        no_improvement = 0
        torch.save({"model": model.state_dict(), "threshold": threshold,
                    "epoch": epoch, "seed": SEED}, run_dir / "best.pt")
        save_table(run_dir / "metrics_val.csv", val_records)
    else:
        no_improvement += 1
    scheduler.step(val_dice)
    if epoch >= 8 and no_improvement >= 6:
        print("Dừng sớm: Dice validation không tăng trong 6 epoch.")
        break

if DEVICE.type == "cuda":
    torch.cuda.synchronize()
save_table(run_dir / "history.csv", history)
config = {"model": "m2", "architecture": "ScratchUNet-32-64-128-256-512",
          "optimizer": "AdamW", "learning_rate": 1e-3, "weight_decay": 1e-4,
          "seed": SEED, "smoke": False, "epochs_run": len(history),
          "best_epoch": best_epoch, "best_threshold": best_threshold,
          "best_val_dice": best_dice, "train_seconds": time.perf_counter() - start,
          "peak_vram_bytes": torch.cuda.max_memory_allocated() if DEVICE.type == "cuda" else 0,
          "parameters": sum(p.numel() for p in model.parameters()),
          "python": sys.version.split()[0], "torch": torch.__version__,
          "numpy": np.__version__, "simpleitk": sitk.Version_VersionString(),
          "matplotlib": matplotlib.__version__,
          "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU"}
(run_dir / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print(f"Checkpoint tốt nhất: epoch {best_epoch}, val Dice {best_dice:.4f}")
print("Đã lưu:", run_dir)


## 5. Điểm validation, biểu đồ học và ảnh dự đoán

Notebook in **Dice, IoU, precision, recall và pixel accuracy** trung bình theo 15 bệnh nhân, từ checkpoint tốt nhất. Pixel accuracy tính cả nền nên có thể cao dù vùng u dự đoán chưa tốt; Dice và IoU là hai số chính để đánh giá phân đoạn. Các hình bên dưới hiện ngay trong notebook và được lưu vào `runs/m2/<seed>/`.


In [ ]:
best = torch.load(run_dir / "best.pt", map_location=DEVICE, weights_only=True)
model.load_state_dict(best["model"])
model.eval()

with (run_dir / "metrics_val.csv").open(encoding="utf-8", newline="") as file:
    validation_scores = list(csv.DictReader(file))
print(f"Validation — {len(validation_scores)} bệnh nhân, checkpoint epoch {best_epoch}:")
for name, label in (("dice", "Dice"), ("iou", "IoU"),
                    ("precision", "Precision"), ("recall", "Recall"),
                    ("pixel_accuracy", "Pixel accuracy")):
    mean_score = np.mean([float(row[name]) for row in validation_scores])
    print(f"  {label}: {mean_score:.4f}")

epochs = [row["epoch"] for row in history]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(epochs, [row["train_loss"] for row in history], "o-")
axes[0].set(xlabel="Epoch", ylabel="Loss", title="Loss train")
axes[1].plot(epochs, [row["val_dice"] for row in history], "o-")
axes[1].set(xlabel="Epoch", ylabel="Dice", title="Dice validation theo bệnh nhân")
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(run_dir / "learning_curve.png", dpi=140)
if IN_NOTEBOOK:
    plt.show()
plt.close(fig)

# Chọn lát có vùng WT rõ để minh họa; chỉ dùng mask để chọn ảnh hiển thị.
index = int(np.argmax(val_data.masks.reshape(len(val_data), -1).sum(axis=1)))
image, truth, case_id = val_data[index]
with torch.inference_mode():
    prediction = torch.sigmoid(model(image[None].to(DEVICE)))[0, 0].cpu().numpy()
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, overlay, title in zip(axes, (None, truth[0].numpy(), prediction >= best_threshold),
                              ("FLAIR", "WT thật", "WT dự đoán")):
    ax.imshow(val_data.images[index], cmap="gray", vmin=0, vmax=1)
    if overlay is not None:
        ax.imshow(np.ma.masked_where(overlay == 0, overlay), cmap="autumn", alpha=0.55)
    ax.set_title(title)
    ax.axis("off")
fig.suptitle(f"{case_id} | ngưỡng {best_threshold:.2f}")
fig.tight_layout()
fig.savefig(run_dir / "preview.png", dpi=140)
if IN_NOTEBOOK:
    plt.show()
plt.close(fig)


## 6. Đánh giá test của M2

Run All tiếp tục dùng **checkpoint và ngưỡng đã chọn từ validation** để đánh giá 15 bệnh nhân test. In Dice, IoU, precision, recall và pixel accuracy; lưu điểm từng bệnh nhân ở `runs/m2/<seed>/metrics_test.csv`. Không dùng điểm test để chọn lại model.


In [ ]:
run_dir = ROOT / "runs" / "m2" / str(SEED)
best = torch.load(run_dir / "best.pt", map_location=DEVICE, weights_only=True)
model.load_state_dict(best["model"])
test_data = SliceDataset(test_cases)
test_loader = DataLoader(test_data, batch_size=16, shuffle=False, num_workers=0)
probabilities, truths = predict_by_patient(test_loader)
records = score_at_threshold(probabilities, truths, float(best["threshold"]))
for row in records:
    row.update(model="m2", seed=SEED, split="test")
save_table(run_dir / "metrics_test.csv", records)
print(f"Test chính thức — {len(records)} bệnh nhân:")
for label, group in (("Tất cả", records),
                     ("HGG", [row for row in records if row["grade"] == "HGG"]),
                     ("LGG", [row for row in records if row["grade"] == "LGG"])):
    print(f"  {label} ({len(group)} ca) | "
          f"Dice {np.mean([row['dice'] for row in group]):.4f} | "
          f"IoU {np.mean([row['iou'] for row in group]):.4f} | "
          f"Precision {np.mean([row['precision'] for row in group]):.4f} | "
          f"Recall {np.mean([row['recall'] for row in group]):.4f} | "
          f"Pixel accuracy {np.mean([row['pixel_accuracy'] for row in group]):.4f}")
print("Đã lưu:", run_dir / "metrics_test.csv")
